# Bring your own CNN: a trained U-Net from checkpoint to Earth Engine

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb)

Runs a U-Net you trained with
[segmentation-models-pytorch](https://github.com/qubvel-org/segmentation_models.pytorch) (SMP)
inside Earth Engine, on your own imagery.

**How to use it**

1. **Try it as it is:** **Runtime → Run all**. It uses a small demo U-Net with random weights
   and sends nothing to Earth Engine. Each cell prints what it did.
2. **Set up your model:** in step 2, fill in your checkpoint, project, input image and region,
   and set `USE_EARTH_ENGINE = True`. **Run all** again: it signs you in and checks your model,
   and stops at step 8 until your weights are uploaded. Nothing is created yet.
3. **Run the Earth Engine steps one at a time, in this order.** Each has a checkbox in its own
   cell: tick it, run that cell, wait until it prints "Done", untick it, and run the cells
   below it (or **Run all** again).
   - step 7: upload the weights (once);
   - step 9: export one test tile;
   - step 10: compare that tile with PyTorch;
   - step 11: export the whole region.

**What it creates in your project:** the weights as table assets (step 5 says how many), and
one image asset each for the test tile and the region.

**Cost:** steps 7 to 11 use your project's Earth Engine quota (EECU time). That is free on a
noncommercial project; a commercial project pays its plan's rate. Their cost has not been
measured for this notebook.

**What has been checked:** TensorEE's building blocks (its versions of each layer) agree with
PyTorch at their validated settings, and one trained ResNet-18 U-Net, prepared this way,
matched PyTorch to 1.2e-5 in Earth Engine on a 64 × 64 input. This tiled recipe has not been
run live as a whole: step 10 checks it for your model. Agreement with PyTorch is not accuracy.

In [ ]:
# @title 1. Install
# Installs from the repository, which gets support for more layers between releases.
%pip install --quiet "tensor-ee[segwater] @ git+https://github.com/noelEOS/tensor-ee"
print("TensorEE is installed.")

## 2. Settings

- Each line says whether you need to change it. As shipped, the notebook runs offline with the
  demo model, and none of these need changing.
- Your input image must already be in Earth Engine, with your model's bands, scaled as in
  training. It must cover the region plus a border on every side (step 6 prints the exact area).

In [ ]:
# Earth Engine
PROJECT = "YOUR_EE_PROJECT"   # change to your Cloud project ID, as in the Code Editor
USE_EARTH_ENGINE = False      # leave False to try the demo offline; True once your model is set below

# Your model: must match how it was trained
CHECKPOINT = None             # change to your saved state_dict, for example "my_unet.pth"
                              # (in Colab, upload it in the Files panel on the left)
ENCODER = "resnet34"          # change to the encoder_name you trained with
BANDS = ("VV", "VH")          # change to your input bands, in training order
CLASSES = 2                   # change to your number of classes

# Your input image and region (the values here are an example in Taiwan)
INPUT_IMAGE = "projects/YOUR_EE_PROJECT/assets/YOUR_PREPARED_INPUT"   # change to your image
CRS = "EPSG:32651"                     # change to a projection in metres, such as your UTM zone
REGION_X, REGION_Y = 286030, 2640860   # change to the region's upper-left corner, in CRS metres
REGION_COLUMNS, REGION_ROWS = 2, 1     # change to the region's size, in tiles of TILE pixels

# Sizes: leave as they are unless your pixels are not 10 m or an export runs out of memory
PIXEL = 10                    # metres per pixel
TILE = 1024                   # side of each region tile, in pixels; larger tiles waste less on borders
TEST_TILE = 256               # side of the test tile, in pixels

# Names for what the Earth Engine steps create: leave as they are, or rename
WEIGHTS_RECORD = "my_unet_weights.json"   # local file listing your uploaded weight tables: keep it
TEST_ASSET = "my_unet_test_tile"          # image asset for the test tile; an existing one is not overwritten
REGION_ASSET = "my_unet_region"           # image asset for the region

if CHECKPOINT is None:
    print("Settings saved. No CHECKPOINT yet, so the notebook uses the demo model, offline.")
elif not USE_EARTH_ENGINE:
    print(f"Settings saved. Model: {CHECKPOINT}. Offline until you set USE_EARTH_ENGINE = True.")
else:
    print(f"Settings saved. Model: {CHECKPOINT}. Earth Engine project: {PROJECT}.")

In [ ]:
# @title 3. Connect to Earth Engine, or stay offline
# @markdown Offline, the notebook prepares every Earth Engine request but sends none. You can
# @markdown change `USE_EARTH_ENGINE` at any time and run from the top again.
import ee
from ee.apitestcase import ApiTestCase


class Stop(Exception):
    """Stops the notebook with a one-line message instead of a traceback."""

    def _render_traceback_(self):
        return [f"Stopped: {self}"]


def live_step(checkbox=None, ticked=True):
    """True when an Earth Engine step should run; otherwise says why it does not."""
    if not USE_EARTH_ENGINE:
        print("Not run: the notebook is offline (USE_EARTH_ENGINE is False in step 2).")
        return False
    if not ticked:
        print(f"Not run: tick {checkbox} above when you are ready, then run this cell.")
        return False
    if not PROJECT or PROJECT == "YOUR_EE_PROJECT":
        raise Stop("set PROJECT to your Cloud project ID in step 2.")
    if CHECKPOINT is None:
        raise Stop("the Earth Engine steps need your trained model: set CHECKPOINT in step 2.")
    return True


# Offline, Earth Engine's own test helper stands in for a signed-in session.
if "offline" in globals():    # undo an earlier offline run in this session
    offline.tearDown()
    ee.data._install_cloud_api_resource = _install_cloud_api_resource
    ee.Reset()
    del offline
if USE_EARTH_ENGINE:
    print("Earth Engine mode: the next cell signs you in.")
else:
    _install_cloud_api_resource = ee.data._install_cloud_api_resource
    offline = ApiTestCase()
    offline.setUp()
    print("Offline mode: nothing will be sent to Earth Engine or created in a project.")

In [ ]:
# @title Sign in (Earth Engine mode only)
# @markdown The first time, follow the link with the Google account you use for Earth Engine.
if live_step():
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
    print(f"Earth Engine is ready — signed in to {PROJECT}.")

In [ ]:
# @title 4. Load the model
# @markdown Without a `CHECKPOINT`, a small demo U-Net with random weights. Your checkpoint must
# @markdown match `ENCODER`, `BANDS` and `CLASSES`: if not, this cell fails with an error listing
# @markdown the weight names that do not fit.
import torch
import segmentation_models_pytorch as smp

torch.set_num_threads(2)


def make_model(**changes):
    settings = dict(encoder_name=ENCODER, encoder_weights=None, in_channels=len(BANDS),
                    classes=CLASSES, activation=None)
    settings.update(changes)
    return smp.Unet(**settings).eval()


if CHECKPOINT is None:
    torch.manual_seed(20261020)
    model = make_model(encoder_depth=3, decoder_channels=(16, 8, 4))
    tile, test_tile = 64, 64      # small tiles for the demo
    print("Loaded the demo model: a small U-Net with random weights.")
else:
    model = make_model()
    model.load_state_dict(torch.load(CHECKPOINT, map_location="cpu", weights_only=True), strict=True)
    tile, test_tile = TILE, TEST_TILE
    print(f"Loaded your model from {CHECKPOINT}.")

In [ ]:
# @title 5. Check the model, its border and its weights
# @markdown - Checks that TensorEE has a building block for every layer of the model.
# @markdown - Each tile is computed from the tile plus an extra border of input on every side
# @markdown   (the halo), so that its edges come out as they would on the whole image; the
# @markdown   border is then dropped. For a ResNet-34 U-Net it is 544 pixels, so a 1,024-pixel
# @markdown   tile reads 2,112 × 2,112.
# @markdown - Counts the weights step 7 uploads. BatchNorm is first merged into the convolutions.
import math

from tensor_ee import check_compatibility, effective_convolutions
from tensor_ee.receptive_field import analyze_receptive_field
from tensor_ee.smp import adapt_smp_model


def prepare(height, width):
    """A copy of the model for one input size; stops unless TensorEE can run every layer."""
    shape = (1, len(BANDS), height, width)
    adapted = adapt_smp_model(model, input_shape=shape)
    report = check_compatibility(adapted, shape)
    if report.status != "supported":
        raise Stop(f"TensorEE cannot run this model yet: {report.to_dict()['blockers'][:3]} "
                   "(notebook 02 explains why a model is refused).")
    return adapted


probe = prepare(64, 64)
support = analyze_receptive_field(probe)
halo, alignment = support.safe_halo, support.alignment
parameters = effective_convolutions(probe)
values = sum(p.weights.size + (0 if p.bias is None else p.bias.size) for p in parameters)
tables = math.ceil(values * 14 / 7_000_000)

print("Supported: TensorEE has a building block for every layer of this model.")
if probe.tensor_ee_smp_metadata["torch_equal"]:
    print("The copy prepared for Earth Engine gives exactly your model's output on a test input.")
print(f"Border (halo): {halo} px ({halo * PIXEL / 1000:g} km) on each side. "
      f"Tile sides must be multiples of {alignment} px.")
print(f"Weights to upload: {values:,} numbers, about {tables} table asset(s).")

In [ ]:
# @title 6. Plan the tiles
# @markdown The test tile is the region's upper-left tile. Each tile reads its border of real input.
from tensor_ee import Grid
from tensor_ee.converter import PixelWindow, plan_tiles


def plan_region(side, columns, rows):
    for name, value in (("tile side", side), ("border", halo)):
        if value % alignment:
            raise Stop(f"the {name} ({value} px) must be a multiple of {alignment} px.")
    padded = Grid(CRS, (PIXEL, 0, REGION_X - halo * PIXEL, 0, -PIXEL, REGION_Y + halo * PIXEL),
                  columns * side + 2 * halo, rows * side + 2 * halo)
    region = PixelWindow(halo, halo, columns * side, rows * side)
    return plan_tiles(padded, side, halo, alignment=alignment, roi=region)


def describe(plan, name):
    context, side = plan.tiles[0].context, plan.core_size[0]
    ratio = sum(t.context.pixels for t in plan.tiles) / plan.roi.pixels
    print(f"{name}: {len(plan.tiles)} tile(s) of {side} × {side} px, each reading "
          f"{context.width} × {context.height} px with its border "
          f"({ratio:.1f} input pixels per output pixel).")


test_plan = plan_region(test_tile, 1, 1)
region_plan = plan_region(tile, REGION_COLUMNS, REGION_ROWS)
describe(test_plan, "Test tile")
describe(region_plan, "Region")
needed = region_plan.global_grid
print(f"Your input image must cover {needed.width} × {needed.height} px from the upper-left "
      f"corner ({needed.transform[2]}, {needed.transform[5]}) in {CRS}.")
if CHECKPOINT is None:
    print("(The demo uses 64 px tiles, not TILE and TEST_TILE.)")

In [ ]:
# @title 7. Upload the weights (once)
# @markdown - Creates the weight table assets, under `projects/PROJECT/assets/tensor_ee_weights_<random>`,
# @markdown   with one export task each, two at a time.
# @markdown - Writes `WEIGHTS_RECORD`, the local list of those tables, which the prediction reads.
# @markdown   Keep a copy: in Colab, download it from the Files panel, since Colab's files are
# @markdown   lost when its runtime is reset.
# @markdown - Cost not measured. For scale, a trained ResNet-18 U-Net (14.3 M numbers) took 29 tables.
UPLOAD_WEIGHTS = False  # @param {type:"boolean"}

import json
import uuid
from pathlib import Path

from tensor_ee import export_weights

if live_step("UPLOAD_WEIGHTS", UPLOAD_WEIGHTS):
    record_path = Path(WEIGHTS_RECORD)
    if record_path.exists():
        raise Stop(f"the weights are already uploaded ({record_path} exists): untick "
                   "UPLOAD_WEIGHTS and go on.")
    namespace = f"projects/{PROJECT}/assets/tensor_ee_weights_{uuid.uuid4().hex[:12]}"
    reported = []

    def progress(record):
        done = sum(shard["status"] == "COMPLETED" for shard in record["shards"])
        if not reported or done != reported[-1]:
            reported.append(done)
            print(f"  {done} of about {tables} table(s) uploaded...", flush=True)

    print("Uploading; follow it here or in the Tasks tab: https://code.earthengine.google.com/tasks")
    export_weights(parameters, namespace, record_path, on_update=progress)
    print(f"Done: the weights are uploaded, and {record_path} lists them. Untick "
          "UPLOAD_WEIGHTS, then run on from step 8.")

In [ ]:
# @title 8. Set up the prediction (nothing runs yet)
# @markdown - Connects the model to your input image and uploaded weights, for the test tile.
# @markdown   Offline, placeholder weights stand in: the prediction is complete but cannot run.
# @markdown - Output bands: `score_k` (the model's raw scores, or logits), `p_k` (probabilities),
# @markdown   `class_index` and, for two classes, `binary_class` (1 where `p_1` ≥ 0.5).
# @markdown - Pixels missing any input band get no prediction.
import json
from pathlib import Path

from tensor_ee import MatchedWeightProvider, Raster, binary_threshold, segmentation_outputs
from tensor_ee.converter import compile_tiled_model
from tensor_ee.converter.request import deep_recursion

SCORES = tuple(f"score_{k}" for k in range(CLASSES))
PROBABILITIES = tuple(f"p_{k}" for k in range(CLASSES))


def weights_record():
    """The uploaded weight tables, or offline placeholders with the same layout."""
    if USE_EARTH_ENGINE and Path(WEIGHTS_RECORD).exists():
        return json.loads(Path(WEIGHTS_RECORD).read_text())
    if USE_EARTH_ENGINE:
        raise Stop(f"{WEIGHTS_RECORD} not found. Upload the weights first (tick UPLOAD_WEIGHTS "
                   "in step 7), or, if you uploaded them before, put that file back in this folder.")
    return {"format": "native_ee_featurecollection_weights_v2", "status": "ready",
            "layers": {p.key: {"shape": list(p.shape), "dtype": p.dtype, "bias": p.bias is not None,
                               "source": p.source, "sha256": p.sha256,
                               "assets": ["projects/YOUR_EE_PROJECT/assets/weights_not_uploaded"]}
                       for p in parameters}}


def input_image():
    if USE_EARTH_ENGINE:
        return ee.Image(INPUT_IMAGE)
    return ee.Image.constant([0] * len(BANDS)).rename(list(BANDS))   # offline stand-in


def build_prediction(plan):
    context = plan.tiles[0].context
    adapted = prepare(context.height, context.width)
    if analyze_receptive_field(adapted).safe_halo != halo:
        raise Stop("the border changed with the input size; please report it as an issue.")
    provider = MatchedWeightProvider(adapted, weights_record())
    source = input_image().select(list(BANDS)).toFloat().reproject(plan.global_grid.projection())
    valid = Raster(source.mask().reduce(ee.Reducer.min()).rename("valid"), ("valid",), plan.global_grid)
    logits = compile_tiled_model(adapted, source, BANDS, plan, weight_provider=provider)
    outputs = segmentation_outputs(logits, logit_bands=SCORES, probability_bands=PROBABILITIES,
                                   validity=valid)
    bands = [outputs.logits.image, outputs.probabilities.image, outputs.classes.image]
    if CLASSES == 2:
        bands.append(binary_threshold(outputs.probabilities, positive_class=1, threshold=0.5,
                                      output_band="binary_class").image)
    return ee.Image.cat(bands).toFloat()   # one type for export; class values stay exact


def export_task(image, plan, asset_name, description):
    grid = plan.roi_grid
    return ee.batch.Export.image.toAsset(
        image=image, description=description, assetId=f"projects/{PROJECT}/assets/{asset_name}",
        region=grid.geometry(), crs=grid.crs, crsTransform=list(grid.transform),
        maxPixels=grid.width * grid.height, pyramidingPolicy={".default": "sample"}, overwrite=False)


with deep_recursion():
    test_prediction = build_prediction(test_plan)
    test_graph_bytes = len(test_prediction.serialize().encode())   # checks it can be sent
if USE_EARTH_ENGINE:
    print("The test-tile prediction is set up with your uploaded weights. Step 9 runs it.")
else:
    print("The test-tile prediction is set up offline, with placeholder weights: complete, "
          "but it cannot run.")

In [ ]:
# @title 9. Export the test tile
# @markdown - Computes the test tile in a batch export and saves it as the image asset `TEST_ASSET`
# @markdown   (the tile only, not its border).
# @markdown - Cost not measured. For scale, notebook 01's 224 × 224 chip used about 65 EECU-seconds.
# @markdown - If it fails for lack of memory or time, lower `TEST_TILE` in step 2 and run from the top.
EXPORT_TEST_TILE = False  # @param {type:"boolean"}

import time

TASKS_URL = "https://code.earthengine.google.com/tasks"


def wait(task, minutes=60):
    """Wait for an export, with a line every 5 minutes; stop if it fails."""
    started, polls = time.time(), 0
    while (state := task.status()["state"]) not in ("COMPLETED", "FAILED", "CANCELLED"):
        if time.time() - started > minutes * 60:
            raise Stop(f"still running after {minutes} minutes. It carries on in Earth Engine: "
                       f"follow it at {TASKS_URL}, and go on once it has finished.")
        time.sleep(30)
        polls += 1
        if polls % 10 == 0:
            print(f"  still running, {polls // 2} min...", flush=True)
    if state != "COMPLETED":
        raise Stop(f"the export ended {state}: {task.status().get('error_message', '')} "
                   f"(see {TASKS_URL})")


if live_step("EXPORT_TEST_TILE", EXPORT_TEST_TILE):
    test_task = export_task(test_prediction, test_plan, TEST_ASSET, "tensor_ee_test_tile")
    test_task.start()
    print(f"Export started; follow it here or at {TASKS_URL}", flush=True)
    wait(test_task)
    print(f"Done: the test tile is saved as projects/{PROJECT}/assets/{TEST_ASSET}. Untick "
          "EXPORT_TEST_TILE, then go on to step 10.")

In [ ]:
# @title 10. Compare the test tile with PyTorch
# @markdown - Run it once step 9 is done. It downloads the test tile's input (with its border)
# @markdown   and the exported scores: 2 small requests, nothing created.
# @markdown - It then runs your model here on the same input and prints the largest difference
# @markdown   (target: at most 1e-4). The input must have no gaps; above 40 MB it is refused.
COMPARE_WITH_PYTORCH = False  # @param {type:"boolean"}

import numpy as np


def download(image, grid, bands):
    names = ("scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY")
    values = ee.data.computePixels({
        "expression": image.select(list(bands)), "fileFormat": "NUMPY_NDARRAY",
        "grid": {"dimensions": {"width": grid.width, "height": grid.height},
                 "affineTransform": dict(zip(names, grid.transform)), "crsCode": grid.crs}})
    return np.stack([values[band] for band in bands]).astype(np.float32)


if live_step("COMPARE_WITH_PYTORCH", COMPARE_WITH_PYTORCH):
    padded = test_plan.global_grid
    size = padded.width * padded.height * (len(BANDS) + 1) * 4
    if size > 40_000_000:
        raise Stop(f"the input is {size / 1e6:.0f} MB; lower TEST_TILE in step 2.")
    source = ee.Image(INPUT_IMAGE).select(list(BANDS)).toFloat().reproject(padded.projection())
    valid = source.mask().reduce(ee.Reducer.min()).rename("valid")
    pixels = download(source.unmask(0).addBands(valid), padded, (*BANDS, "valid"))
    if pixels[-1].min() < 1:
        raise Stop("the input has gaps in the test tile or its border; move the region or fill them.")
    exported = download(ee.Image(f"projects/{PROJECT}/assets/{TEST_ASSET}"), test_plan.roi_grid, SCORES)

    with torch.no_grad():
        reference = model(torch.from_numpy(pixels[None, :-1].copy()))[0].numpy()
    core = reference[:, halo:halo + test_tile, halo:halo + test_tile]
    error = np.abs(exported - core)
    print(f"Largest difference in the scores: {error.max():.3e} (target 1e-4); "
          f"mean {error.mean():.3e}; pixels with a different class: "
          f"{int((exported.argmax(0) != core.argmax(0)).sum())} of {core[0].size}.")
    print("Done: within the target. Earth Engine agrees with PyTorch on this tile." if error.max() <= 1e-4
          else "Done, but above the target: please report it as an issue, with these numbers.")

In [ ]:
# @title 11. Export the region
# @markdown - Run it once step 10 is within the target. One batch export, saved as the image
# @markdown   asset `REGION_ASSET` (the region only, not its border).
# @markdown - Cost, memory use and time not measured at these tile sizes.
EXPORT_REGION = False  # @param {type:"boolean"}

if live_step("EXPORT_REGION", EXPORT_REGION):
    with deep_recursion():
        region_task = export_task(build_prediction(region_plan), region_plan, REGION_ASSET,
                                  "tensor_ee_region")
        region_task.start()
    print(f"Export started; follow it here or at {TASKS_URL}", flush=True)
    wait(region_task, minutes=240)
    print(f"Done: the region is saved as projects/{PROJECT}/assets/{REGION_ASSET}.")

## Good to know

- **Coming back later:** put your `WEIGHTS_RECORD` file back next to the notebook (in Colab,
  upload it in the Files panel, like the checkpoint), set
  `USE_EARTH_ENGINE = True`, and **Run all** with every checkbox unticked; then tick only the
  step you need.
- **Evidence:** step 10 is agreement in Earth Engine for your model on one tile; keep its
  numbers. Accuracy is separate: evaluate the exports against your own labelled data.
- **Details:** [the guide this notebook follows](https://github.com/noelEOS/tensor-ee/blob/main/docs/own-model.md)
  covers other encoders, irregular regions and masks of valid pixels.
- **Clean up:** the weight tables and images stay in your project until you delete them (Code
  Editor, **Assets** tab). Keep the weight tables while you predict with them.